# Chapter 10: From Grounded Evidence to a Defensible Report

**The question.** A validation report can get every number right and still mislead, for example by leaving out an unfavorable result or the conditions a finding depends on. How do we make sure a report says everything it must? This chapter fixes the required content before anyone writes, then checks the report against that list.

**The steps.**

1. Write down what the report section must contain: 25 facts from the evidence store and 6 required caveats. This list is the *evidence packet*.
2. Check a proposed outline (a *content plan*) against the packet before any text is written. Incomplete outlines are rejected.
3. Assemble the section from the store, so every number carries the address of the file it came from.
4. Check the finished section: are all 25 facts and all 6 caveats present?

The section reports Chapter 5's comparison: the LLM-proposed two-cluster rule against continued search with the old rule, confirmed in Chapter 9.

**Result.** The complete outline passes. An outline missing one interval, or missing the caveat that states the negative result, is rejected before writing starts. The finished section contains all 25 facts and all 6 caveats.

**Map of this notebook.**

| Step | Question | Where to look |
|---|---|---|
| 1 | What must the section contain? | the two printed counts and the caveat texts |
| 2 | Is a proposed outline complete? | `accepted` column |
| 3 | Where does each number come from? | the `[source: ...]` part of each line |
| 4 | Does the finished section contain everything? | the "present" counts |

**Terms.** *Fact identifier*: the name of one stored number, such as `region:confirmation/continuation_73101/contrast`. *Qualification*: a required caveat about scope or limits. *Continuation*: the old one-cluster search run longer. *Reflection*: the search with the LLM's two-cluster rule. *Contrast*: how much worse the model does inside a group than outside it. The customers are simulated.

The notebook makes no language-model call. The recorded experiment in which a language model drafted this section is in `regional_live/ch10_report.ipynb`. The store is `book/evidence/parallel_aml_20260921/store.json`. Run the cells in order; the `assert` lines check which outlines are accepted and that the finished section is complete.

## Setup and source checks

Loads the saved results and checks that the files have not changed. Select the kernel of the environment where `aiv` is installed. The packet, outline check and assembly functions come from `companion.py` and use the Chapter 7 verifier.

In [1]:
from pathlib import Path
import os
import platform
import sys
import importlib.util

# Locate the data root: the directory whose relative layout the evidence records use.
MARKER = "book/notebooks/regional_case/companion.py"

def find_data_root():
    configured = os.environ.get("MODEL_VALIDATION_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / MARKER).is_file():
            raise FileNotFoundError(f"MODEL_VALIDATION_ROOT must contain {MARKER}")
        return root, "MODEL_VALIDATION_ROOT"
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / MARKER).is_file():
            return path, "repository checkout"
    try:
        from aiv.companions import locate
    except ImportError:
        raise FileNotFoundError(
            "Companion data not found. Install the aiv package, open the notebook "
            "inside the repository checkout, or set MODEL_VALIDATION_ROOT.") from None
    return locate()

ROOT, DATA_SOURCE = find_data_root()
HELPERS = ROOT / "book/notebooks/regional_case"
for path in (ROOT, ROOT / "book", HELPERS):
    value = str(path)
    if value in sys.path:
        sys.path.remove(value)
    sys.path.insert(0, value)

# Load current helper files even if this kernel cached an older notebook revision.
def load_helper_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    try:
        spec.loader.exec_module(module)
    except Exception:
        sys.modules.pop(name, None)
        raise
    return module

load_helper_module("companion", HELPERS / "companion.py")
load_helper_module("ch02_helpers", HELPERS / "ch02_helpers.py")
load_helper_module("weighted_leaf_geometry", ROOT / "book/weighted_leaf_geometry.py")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
print("Python:", platform.python_version())
print("Data root found from:", DATA_SOURCE)
from companion import RUN, read, load, frontier, confirmation, measure, assignments
pd.set_option("display.float_format", "{:.6f}".format)
protocol, results, frozen = load()
print("Evidence:", RUN.relative_to(ROOT).as_posix())
print("Mode: saved-evidence replay; no fitting or live LLM calls")


Python: 3.12.3
Data root found from: repository checkout
Evidence: book/evidence/aml_medoid_loop_leaf_output_20260921
Mode: saved-evidence replay; no fitting or live LLM calls


## Step 1: Decide what the section must contain

The evidence packet lists, before any writing, every fact the section must report and every caveat it must state (Sections 10.1 and 10.3). `report_packet()` builds the packet for the Chapter 5 comparison, and `QUALIFIERS` holds the six caveat texts. The cell prints how many facts and caveats are required, then shows both lists.

In [2]:
from companion import report_packet, QUALIFIERS, validate_plan, render_report
store,packet=report_packet()
print("Required facts:",len(packet["required_facts"]))
print("Required qualifications:",len(packet["required_qualifiers"]))
display(packet)
display(QUALIFIERS)

Required facts: 25
Required qualifications: 6


{'required_facts': ['region:protocol/confirmation_n',
  'region:confirmation/continuation_73101/brier',
  'region:confirmation/continuation_73101/share',
  'region:confirmation/continuation_73101/contrast',
  'region:confirmation/reflection_73101/brier',
  'region:confirmation/reflection_73101/share',
  'region:confirmation/reflection_73101/contrast',
  'region:comparison/reflection-continuation_73101/contrast_difference',
  'region:comparison/reflection-continuation_73101/simultaneous_ci',
  'region:confirmation/continuation_73102/brier',
  'region:confirmation/continuation_73102/share',
  'region:confirmation/continuation_73102/contrast',
  'region:confirmation/reflection_73102/brier',
  'region:confirmation/reflection_73102/share',
  'region:confirmation/reflection_73102/contrast',
  'region:comparison/reflection-continuation_73102/contrast_difference',
  'region:comparison/reflection-continuation_73102/simultaneous_ci',
  'region:confirmation/continuation_73103/brier',
  'region:co

{'synthetic': 'These measurements concern simulated account-months and do not establish operational AML adequacy.',
 'fixed': 'The logistic predictor and auxiliary error model remained frozen; search changed region membership.',
 'conditional': 'Intervals are approximate simultaneous 95% normal intervals for 19 claims, conditional on the fitted models and frozen regions.',
 'replication': 'Three optimizer seeds share one discovery sample, one LLM proposal and one confirmation sample.',
 'comparison': 'All three reflection-minus-continuation intervals include zero; a positive advantage is unresolved.',
 'coverage': 'The selected regions differ in coverage; contrast is not predictor repair.'}

**Read the output.** **In one line: the section must report 25 facts and 6 caveats, including the unfavorable ones.**

**First display (the packet).** `required_facts` lists the 25 fact names: the confirmation sample size, then 8 facts for each of the 3 seeds. For each seed that is the error (`brier`), size (`share`) and `contrast` of the continuation group and of the reflection group, plus their difference (`contrast_difference`) and its plausible range (`simultaneous_ci`). `required_qualifiers` lists the 6 caveat names.

**Second display (the caveats).** Each caveat name with its exact text: `synthetic` (simulated customers), `fixed` (the models stayed frozen), `conditional` (what the intervals assume), `replication` (the seeds share their data and the proposal), `comparison` (no advantage was found) and `coverage` (the groups differ in size; contrast is not a repair of the model).

**Notice.** The control group (continuation) is required next to the LLM rule, so a section cannot report the new rule alone. Seed 73102, where the LLM rule did worse, is required too.

## Step 2: Check an outline before writing

A content plan is an outline: the facts and caveats a writer proposes to include. `validate_plan` rejects an outline that leaves something out, adds something not in the packet or repeats an entry, and it checks every planned fact against the store with the Chapter 7 verifier. The cell tests three hand-written outlines: a complete one, one missing the last fact (the seed-73103 interval) and one missing the `comparison` caveat. None of them came from a language model.

In [3]:
from copy import deepcopy
complete={"facts":packet["required_facts"].copy(),"qualifiers":packet["required_qualifiers"].copy()}
missing_fact=deepcopy(complete);missing_fact["facts"].pop()
missing_scope=deepcopy(complete);missing_scope["qualifiers"].remove("comparison")
rows=[]
for name,plan in [("complete",complete),("missing interval",missing_fact),("missing comparison qualifier",missing_scope)]:
    valid,reason=validate_plan(store,packet,plan)
    rows.append(dict(plan=name,accepted=valid,reason=reason))
assert [r["accepted"] for r in rows]==[True,False,False]
with pd.option_context("display.max_colwidth",None):
    display(pd.DataFrame(rows))

,plan,accepted,reason
0,complete,True,Complete source-checked packet
1,missing interval,False,Missing or unexpected evidence/qualification identifiers
2,missing comparison qualifier,False,Missing or unexpected evidence/qualification identifiers


**Read the output.** **In one line: only the complete outline passes; both incomplete outlines are rejected before any text is written.**

**Columns.** `plan`: which outline. `accepted`: whether it passed. `reason`: why.

**Notice.** The most important rejection is the outline missing the `comparison` caveat. Without it, a section could report the small positive differences and the new rule without saying that no advantage was found. The check enforces the packet; it does not show whether a writer, human or model, would propose a complete outline unprompted.

## Step 3: Assemble the section from the store

Once an outline is accepted, the section is built straight from the store, so no number is retyped. Each line gives the fact's name, its stored value and the file and position it came from; the six caveats follow in full. Values are printed unrounded, as stored; a formatted report would round them for display and keep the addresses.

In [4]:
report=render_report(store,packet,complete)
print(report)

Regional discovery report: deterministic assembly from an authored content plan.

region:protocol/confirmation_n: 200000 [source: book/evidence/aml_medoid_loop_leaf_output_20260921/protocol.json#/confirmation_n]
region:confirmation/continuation_73101/brier: 0.07258430947407625 [source: book/evidence/aml_medoid_loop_leaf_output_20260921/results.json#/confirmation/continuation_73101/brier]
region:confirmation/continuation_73101/share: 0.113925 [source: book/evidence/aml_medoid_loop_leaf_output_20260921/results.json#/confirmation/continuation_73101/share]
region:confirmation/continuation_73101/contrast: 0.057939624355884026 [source: book/evidence/aml_medoid_loop_leaf_output_20260921/results.json#/confirmation/continuation_73101/contrast]
region:confirmation/reflection_73101/brier: 0.07285078874096361 [source: book/evidence/aml_medoid_loop_leaf_output_20260921/results.json#/confirmation/reflection_73101/brier]
region:confirmation/reflection_73101/share: 0.11522 [source: book/evidence/aml_m

**Read the output.** **In one line: every number in the section points to the exact file and position it came from.**

**Reading a line.** `fact name: value [source: file#/position]`. For example, the continuation contrast at seed 73101, 0.057939624355884026, comes from `results.json#/confirmation/continuation_73101/contrast`.

**Notice.** A reviewer can check any line against the evidence, as in Chapter 6. This output is a structured record of what the section must say, not readable prose. Turning it into paragraphs is the drafting step that Chapter 10 gives to a language model under a closed-world writing contract (Section 10.4), and the draft must then be checked again.

## Step 4: Check the finished section

A report is checked as delivered, not only as planned (Section 10.10). The cell counts how many of the 25 fact names and 6 caveats appear in the assembled section, then prints the caveat that carries the study's negative result.

In [5]:
facts_found=sum(id_ in report for id_ in packet["required_facts"])
qualifiers_found=sum(QUALIFIERS[q] in report for q in packet["required_qualifiers"])
print(f"Fact identifiers present: {facts_found} of {len(packet['required_facts'])}")
print(f"Qualifications present: {qualifiers_found} of {len(packet['required_qualifiers'])}")
assert facts_found==len(packet["required_facts"]) and qualifiers_found==len(packet["required_qualifiers"])
print("Comparison qualification:",QUALIFIERS["comparison"])

Fact identifiers present: 25 of 25
Qualifications present: 6 of 6
Comparison qualification: All three reflection-minus-continuation intervals include zero; a positive advantage is unresolved.


**Read the output.** **In one line: the finished section contains all 25 facts and all 6 caveats.**

**Notice.** This is a coverage check: it confirms the required evidence is cited, but not that the sentences around the citations interpret it correctly. In the live drafting experiment (`regional_live/ch10_report.ipynb`), six of nine language-model drafts passed this kind of check, while a separate review found that seven of the nine needed substantive revision.

## What we learned

Before anything was written, the report section was pinned down as 25 facts and 6 caveats. Outlines that dropped an interval or the caveat carrying the negative result were rejected, the section was assembled with a source address for every number, and the finished section was checked for completeness.

These checks make sure the right evidence is present and traceable. They cannot tell whether written sentences describe that evidence faithfully; that needs a separate review, which the live experiment shows is necessary.


## What to try

1. **Drop the control.** Remove `region:confirmation/continuation_73102/contrast` from the complete outline and confirm it is rejected. How could the section mislead without it?
2. **Add an extra fact.** Add `region:discovery/reflection_73101/region/contrast` to the complete outline. Predict the result before running it. Why is a discovery number risky in a section about confirmation?
3. **Write one sentence.** Summarize seed 73102 in one sentence using the assembled lines. Check each number with `verify`. What about the sentence can the verifier not check?
4. **Plan a live test.** Choose two writing instructions to compare and two held-back packets to test them on. Which results could be measured automatically, and which would need a human review?
